# LeetCode 360 — Sort Transformed Array bằng Python

Link đề: https://leetcode.com/problems/sort-transformed-array/ · Độ khó: **Medium** · Chủ đề: Array, Math, Two Pointers (hai con trỏ), Sorting (sắp xếp)

> **Bài Premium (🔒).** Đề trong notebook này được viết theo hiểu biết của mình về bài và theo bản mô tả đã có; nếu bạn có tài khoản Premium, hãy **đối chiếu lại đề gốc** (nhất là ví dụ và ràng buộc) trước khi nộp.

Skeleton LeetCode:

```python
class Solution:
    def sortTransformedArray(self, nums: List[int], a: int, b: int, c: int) -> List[int]:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án trong notebook đã được kiểm chứng bằng code (đối chiếu bốn cách làm với oracle `sorted(f(x) for x in nums)` trên hàng nghìn input ngẫu nhiên đúng ràng buộc). Ví dụ nào ghi "tự dựng" là do mình thêm để minh họa — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho mảng số nguyên `nums` **đã sắp tăng** và ba số nguyên `a, b, c`; áp dụng hàm bậc hai `f(x) = a·x² + b·x + c` lên **từng phần tử**, rồi trả về mảng kết quả **đã sắp tăng**. Follow-up: làm trong **`O(n)`**.
2. Nắm các công cụ Python: định nghĩa **hàm lồng** `def f(x): ...`, `sorted`, **hai con trỏ** `i, j` kết hợp **điền kết quả từ cuối (hoặc từ đầu)** vào mảng `ans`, **trộn hai dãy đã sắp** (merge), `bisect`, đảo mảng `[::-1]`, và mẹo **đối xứng dấu** (`−f`).
3. Đi từ cách **tính `f` rồi sắp xếp** (`O(n log n)`) → **hai con trỏ hai đầu** (`O(n)`) → **tách tại đỉnh parabol rồi trộn** (`O(n)`) → **đếm (counting sort)** (khi miền giá trị nhỏ).
4. Hiểu mấu chốt: `f` là **parabol** — dãy giá trị `f(nums[i])` trên mảng đã sắp là **"chữ U" (hoặc "chữ ∩")**: giảm rồi tăng (khi `a > 0`) hoặc tăng rồi giảm (khi `a < 0`). Do đó giá trị **lớn nhất** (khi `a > 0`) hoặc **nhỏ nhất** (khi `a < 0`) luôn nằm ở **một trong hai đầu** của đoạn đang xét ⇒ hai con trỏ nhặt dần từ hai đầu.
5. Tránh các lỗi kinh điển: chỉ nghĩ tới `a > 0`, so sánh `|x|` thay vì `f(x)`, điền **sai chiều** (từ đầu thay vì từ cuối khi `a > 0`), dùng `while i < j` làm **bỏ sót phần tử cuối**, và quên các trường hợp `a = 0` hoặc `a < 0`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên **đã sắp tăng** `nums` và ba số nguyên `a`, `b`, `c`. Áp dụng hàm bậc hai

> `f(x) = a·x² + b·x + c`

lên **mỗi phần tử** `nums[i]`, rồi trả về **mảng kết quả theo thứ tự tăng dần**.

**Ví dụ 1**

- Input: `nums = [-4,-2,2,4], a = 1, b = 3, c = 5`
- Output: `[3,9,15,33]`

**Ví dụ 2**

- Input: `nums = [-4,-2,2,4], a = -1, b = 3, c = 5`
- Output: `[-23,-5,1,7]`

**Ràng buộc:**

- `1 <= nums.length <= 200`
- `-100 <= nums[i], a, b, c <= 100`
- `nums` được sắp **tăng dần**.

**Follow-up:** bạn có giải được trong thời gian `O(n)` không?

In [1]:
import random
import time
from bisect import bisect_left
from collections import Counter
from heapq import merge
from typing import List

examples = [
    (([-4, -2, 2, 4], 1, 3, 5), [3, 9, 15, 33]),
    (([-4, -2, 2, 4], -1, 3, 5), [-23, -5, 1, 7]),
]
for (nums, a, b, c), expected in examples:
    print(nums, a, b, c, "-> kỳ vọng", expected)

[-4, -2, 2, 4] 1 3 5 -> kỳ vọng [3, 9, 15, 33]
[-4, -2, 2, 4] -1 3 5 -> kỳ vọng [-23, -5, 1, 7]


## 3. Kiến thức nền

### 3.1. Hàm bậc hai và bảng giá trị

`f(x) = a·x² + b·x + c`. Với mỗi `x` ta thay vào: ví dụ `a = 1, b = 3, c = 5`, `x = −4`: `1·16 + 3·(−4) + 5 = 9`. Hàm trong Python: `def f(x): return a * x * x + b * x + c`. Hàm **lồng** trong hàm khác (như bên trong `sortTransformedArray`) vẫn "nhìn thấy" `a, b, c` của hàm ngoài.

In [2]:
a, b, c = 1, 3, 5

def f(x):
    return a * x * x + b * x + c

for x in [-4, -2, 2, 4]:
    print(f"f({x}) = {a}*{x}^2 + {b}*{x} + {c} = {f(x)}")

f(-4) = 1*-4^2 + 3*-4 + 5 = 9
f(-2) = 1*-2^2 + 3*-2 + 5 = 3
f(2) = 1*2^2 + 3*2 + 5 = 15
f(4) = 1*4^2 + 3*4 + 5 = 33


### 3.2. Hình dạng parabol: chữ U, chữ ∩ hay đường thẳng

- `a > 0`: parabol **mở lên** (chữ U): giá trị giảm tới **đỉnh** rồi tăng.
- `a < 0`: parabol **mở xuống** (chữ ∩): giá trị tăng tới đỉnh rồi giảm.
- `a = 0`: **đường thẳng** `f(x) = b·x + c`: tăng nếu `b > 0`, giảm nếu `b < 0`, hằng nếu `b = 0`.

Đỉnh ở `x* = −b / (2a)`. Cách thấy "tăng hay giảm" mà không cần số thực: **hiệu hai giá trị liên tiếp** `f(x + 1) − f(x) = a·(2x + 1) + b` — dấu của nó cho biết đang tăng hay giảm tại `x`.

In [3]:
def table(a, b, c, xs):
    return [a * x * x + b * x + c for x in xs]

xs = list(range(-5, 6))
print("a>0 (chữ U)  :", table(1, 0, 0, xs))      # 25 16 9 4 1 0 1 4 9 16 25
print("a<0 (chữ ∩) :", table(-1, 0, 0, xs))
print("a=0, b>0     :", table(0, 2, 1, xs))
print("a=0, b<0     :", table(0, -2, 1, xs))
a, b = 1, 3
print("đỉnh x* =", -b / (2 * a), "| f(x+1)-f(x) tại x=-2 là", a * (2 * -2 + 1) + b, "(>0 nghĩa là đã tăng)")

a>0 (chữ U)  : [25, 16, 9, 4, 1, 0, 1, 4, 9, 16, 25]
a<0 (chữ ∩) : [-25, -16, -9, -4, -1, 0, -1, -4, -9, -16, -25]
a=0, b>0     : [-9, -7, -5, -3, -1, 1, 3, 5, 7, 9, 11]
a=0, b<0     : [11, 9, 7, 5, 3, 1, -1, -3, -5, -7, -9]
đỉnh x* = -1.5 | f(x+1)-f(x) tại x=-2 là 0 (>0 nghĩa là đã tăng)


### 3.3. Hai con trỏ và điền kết quả từ cuối

Với `i = 0` (đầu) và `j = n − 1` (cuối), mỗi bước ta **chọn** một trong hai đầu rồi **dịch** con trỏ tương ứng vào trong. Nếu mỗi lần chọn "giá trị **lớn nhất** còn lại" thì kết quả được **điền vào vị trí `k` từ cuối về đầu** (`k = n − 1, n − 2, ...`); nếu chọn "giá trị **nhỏ nhất** còn lại" thì điền từ đầu (`k = 0, 1, ...`). Vòng lặp chạy khi `i <= j` (còn ít nhất một phần tử).

In [4]:
vals = [9, 3, 15, 33]          # giá trị f ở ví dụ 1 theo thứ tự nums (chữ U: giảm rồi tăng)
i, j = 0, len(vals) - 1
out = [0] * len(vals)
k = len(vals) - 1
while i <= j:
    if vals[i] >= vals[j]:
        out[k] = vals[i]
        i += 1
    else:
        out[k] = vals[j]
        j -= 1
    k -= 1
print(out)                      # [3, 9, 15, 33]

[3, 9, 15, 33]


### 3.4. Trộn hai dãy đã sắp (merge)

Cho hai dãy tăng `A`, `B`, ta tạo dãy tăng chứa tất cả: dùng hai con trỏ, mỗi lần lấy phần tử **nhỏ hơn** ở đầu hai dãy. Thư viện: `heapq.merge(A, B)` (trả iterator, dùng `list(...)`). Tốn `O(|A| + |B|)`.

In [5]:
def merge_two(A, B):
    out, i, j = [], 0, 0
    while i < len(A) and j < len(B):
        if A[i] <= B[j]:
            out.append(A[i]); i += 1
        else:
            out.append(B[j]); j += 1
    out.extend(A[i:])
    out.extend(B[j:])
    return out

print(merge_two([1, 4, 9], [2, 3, 10, 11]))
print(list(merge([1, 4, 9], [2, 3, 10, 11])))

[1, 2, 3, 4, 9, 10, 11]
[1, 2, 3, 4, 9, 10, 11]


### 3.5. `bisect_left`, đảo dãy và đối xứng dấu

- `bisect_left(lst, v)`: vị trí đầu tiên mà `lst[pos] >= v` trong **dãy đã sắp tăng** — tìm kiếm nhị phân `O(log n)`.
- `lst[::-1]`: đảo ngược danh sách (một dãy giảm thành dãy tăng).
- **Đối xứng dấu:** nếu `g(x) = −f(x)` thì thứ tự của `g` **ngược** thứ tự của `f`: sắp `g` tăng rồi **đổi dấu và đảo** sẽ thành `f` sắp tăng. Dùng để đưa trường hợp `a < 0` về `a > 0`.

In [6]:
print(bisect_left([1, 3, 5, 7], 5), bisect_left([1, 3, 5, 7], 4))       # 2 2
vals = [9, 3, 15, 33]
neg = sorted(-v for v in vals)                  # sắp g = -f tăng
print(neg, "->", [-v for v in reversed(neg)])    # đổi dấu và đảo: ra f sắp tăng
print(sorted(vals))

2 2
[-33, -15, -9, -3] -> [3, 9, 15, 33]
[3, 9, 15, 33]


### 3.6. `sorted` — cách đơn giản nhất

`sorted(iterable)` trả list mới đã sắp tăng, tốn `O(n log n)`. Đây là baseline: tính `f` cho từng phần tử rồi `sorted`. Mọi cách còn lại cố gắng tận dụng việc `nums` **đã sắp** để đạt `O(n)`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Ví dụ 1:** `nums = [-4,-2,2,4]`, `a = 1, b = 3, c = 5`. Tính `f` cho từng phần tử:

| `x` | `f(x) = x² + 3x + 5` |
|---|---|
| `−4` | `16 − 12 + 5 = 9` |
| `−2` | `4 − 6 + 5 = 3` |
| `2` | `4 + 6 + 5 = 15` |
| `4` | `16 + 12 + 5 = 33` |

Dãy theo thứ tự `nums`: `[9, 3, 15, 33]` — **giảm rồi tăng** (chữ U), đáy ở `x = −2` (đỉnh lý thuyết `−1.5`). Sắp lại: `[3, 9, 15, 33]`.

**Ví dụ 2:** `a = −1, b = 3, c = 5`: `f(−4) = −16 − 12 + 5 = −23`, `f(−2) = −4 − 6 + 5 = −5`, `f(2) = −4 + 6 + 5 = 7`, `f(4) = −16 + 12 + 5 = 1`. Dãy `[−23, −5, 7, 1]` — **tăng rồi giảm** (chữ ∩). Sắp: `[−23, −5, 1, 7]`.

**Quan sát:** trong chữ U, giá trị **lớn nhất** luôn ở **một trong hai đầu**; sau khi lấy nó, phần còn lại vẫn là chữ U (hoặc chỉ một nhánh), nên lại lớn nhất ở một trong hai đầu. Cứ thế nhặt từ hai đầu vào. Chữ ∩ thì nhặt giá trị **nhỏ nhất** từ hai đầu.

In [7]:
for (nums, a, b, c), expected in examples:
    vals = [a * x * x + b * x + c for x in nums]
    shape = "chữ U (giảm rồi tăng)" if a > 0 else "chữ ∩ (tăng rồi giảm)" if a < 0 else "đường thẳng"
    print(f"a = {a:2d}: f theo thứ tự nums = {vals}  -> {shape}; sắp: {sorted(vals)}")

a =  1: f theo thứ tự nums = [9, 3, 15, 33]  -> chữ U (giảm rồi tăng); sắp: [3, 9, 15, 33]
a = -1: f theo thứ tự nums = [-23, -5, 7, 1]  -> chữ ∩ (tăng rồi giảm); sắp: [-23, -5, 1, 7]


## 5. Cách 1 — Brute Force / baseline: tính `f` rồi sắp xếp

**Ý tưởng.** Làm đúng đề: tính `f(x)` cho mọi `x` rồi `sorted`.

In [8]:
def sort_transformed_sorted(nums, a, b, c):
    return sorted(a * x * x + b * x + c for x in nums)


for (nums, a, b, c), expected in examples:
    print(sort_transformed_sorted(nums, a, b, c), "(kỳ vọng", expected, ")")

[3, 9, 15, 33] (kỳ vọng [3, 9, 15, 33] )
[-23, -5, 1, 7] (kỳ vọng [-23, -5, 1, 7] )


**Độ phức tạp (Cách 1).** Tính `f` mất `O(n)`, sắp xếp `O(n log n)` ⇒ **Time `O(n log n)`**; **Space `O(n)`** (mảng kết quả). Với `n <= 200` rất nhanh. Đây cũng là **oracle** (đáp án chuẩn) để đối chiếu các cách khác.

**Nút thắt (theo follow-up):** ta chưa dùng thông tin `nums` đã sắp, nên phải trả giá `log n`. Bài hỏi cách `O(n)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Hàm bậc hai trên dãy đã sắp là "chữ U" hoặc "chữ ∩".**

- `a > 0`: dãy `f(nums[i])` giảm tới đỉnh rồi tăng. Với đoạn `[i, j]` đang xét, giá trị **lớn nhất** nằm ở `i` hoặc `j` (hàm lồi: lớn nhất trên đoạn ở biên). Nên so `f(nums[i])` với `f(nums[j])`, lấy cái lớn hơn đặt vào **cuối** mảng kết quả, rồi thu đoạn lại.
- `a < 0`: hàm lõm: giá trị **nhỏ nhất** ở biên. So hai đầu, lấy cái **nhỏ hơn** đặt vào **đầu** mảng kết quả.
- `a = 0`: tuyến tính; cả hai đầu đều là cực trị — nhánh `a >= 0` (nhặt lớn nhất từ biên điền từ cuối) vẫn đúng: nếu `b >= 0` luôn lấy đầu phải; nếu `b < 0` luôn lấy đầu trái.

Mỗi bước loại một phần tử ⇒ `n` bước, mỗi bước `O(1)` ⇒ **`O(n)`**. Đó là Cách 2.

Một cách nhìn khác (Cách 3): **tách tại đỉnh**: hai nhánh (trái đỉnh và phải đỉnh) mỗi nhánh đơn điệu; đảo nhánh giảm thành tăng rồi **trộn** hai dãy tăng — cũng `O(n)`.

## 7. Các cách tiếp theo

### Cách 2 — Hai con trỏ hai đầu, điền từ cuối (`a >= 0`) hoặc từ đầu (`a < 0`)

**Giải thích từng đoạn.**

1. Định nghĩa `f(x)`; `n = len(nums)`; `ans = [0] * n`; `i, j = 0, n - 1`.
2. Nếu `a >= 0`: `k = n - 1` (điền từ cuối). Trong khi `i <= j`: tính `fi, fj`; nếu `fi >= fj` thì `ans[k] = fi; i += 1` ngược lại `ans[k] = fj; j -= 1`; rồi `k -= 1`.
3. Nếu `a < 0`: `k = 0` (điền từ đầu). Trong khi `i <= j`: nếu `fi <= fj` thì lấy `fi` (nhỏ hơn) ngược lại lấy `fj`; rồi `k += 1`.
4. Trả `ans`.

Với `i == j`, `fi == fj` nên lấy một trong hai, thu về `i > j`, kết thúc.

In [9]:
def sort_transformed_two_pointers(nums, a, b, c):
    def f(x):
        return a * x * x + b * x + c

    n = len(nums)
    ans = [0] * n
    i, j = 0, n - 1
    if a >= 0:                          # chữ U (hoặc đường thẳng): nhặt giá trị LỚN nhất, điền từ cuối
        k = n - 1
        while i <= j:
            fi, fj = f(nums[i]), f(nums[j])
            if fi >= fj:
                ans[k] = fi
                i += 1
            else:
                ans[k] = fj
                j -= 1
            k -= 1
    else:                               # chữ ∩: nhặt giá trị NHỎ nhất, điền từ đầu
        k = 0
        while i <= j:
            fi, fj = f(nums[i]), f(nums[j])
            if fi <= fj:
                ans[k] = fi
                i += 1
            else:
                ans[k] = fj
                j -= 1
            k += 1
    return ans


for (nums, a, b, c), expected in examples:
    print(sort_transformed_two_pointers(nums, a, b, c), "(kỳ vọng", expected, ")")

[3, 9, 15, 33] (kỳ vọng [3, 9, 15, 33] )
[-23, -5, 1, 7] (kỳ vọng [-23, -5, 1, 7] )


In [10]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionTwoPointers:
    def sortTransformedArray(self, nums: List[int], a: int, b: int, c: int) -> List[int]:
        def f(x: int) -> int:
            return a * x * x + b * x + c

        n = len(nums)
        ans = [0] * n
        i, j = 0, n - 1
        if a >= 0:
            k = n - 1
            while i <= j:
                fi, fj = f(nums[i]), f(nums[j])
                if fi >= fj:
                    ans[k] = fi
                    i += 1
                else:
                    ans[k] = fj
                    j -= 1
                k -= 1
        else:
            k = 0
            while i <= j:
                fi, fj = f(nums[i]), f(nums[j])
                if fi <= fj:
                    ans[k] = fi
                    i += 1
                else:
                    ans[k] = fj
                    j -= 1
                k += 1
        return ans


print(SolutionTwoPointers().sortTransformedArray([-4, -2, 2, 4], -1, 3, 5))      # [-23, -5, 1, 7]

[-23, -5, 1, 7]


**Độ phức tạp Cách 2:** mỗi vòng loại một phần tử, `n` vòng, mỗi vòng `O(1)` (hai lần tính `f`) ⇒ **Time `O(n)`**; **Space `O(n)`** cho `ans` (không tính bộ nhớ phụ nào khác — `O(1)` phụ).

### Cách 3 — Tách tại đỉnh rồi trộn hai nhánh đơn điệu

**Ý tưởng.** Với `a > 0`: bên trái đỉnh `x* = −b/(2a)` hàm **giảm**, bên phải **tăng**. Tách `nums` thành hai phần tại vị trí `p` (phần tử đầu tiên mà `f` bắt đầu tăng). Nhánh trái có giá trị giảm ⇒ **đảo** thành tăng; nhánh phải đã tăng; **trộn** hai dãy tăng.

Tìm `p` bằng số nguyên (không dùng số thực): `f(x + 1) − f(x) = a(2x + 1) + b`; hàm "đang đi xuống rồi" khi `2a·x + b < 0`. Dãy `key[i] = 2·a·nums[i] + b` **tăng theo `i`** (vì `a > 0` và `nums` tăng) ⇒ `p = bisect_left(key, 0)` là vị trí đầu tiên có `2a·x + b >= 0` (từ đó hàm không giảm nữa). Nhánh trái `nums[:p]`, nhánh phải `nums[p:]`.

Các trường hợp còn lại:

- `a = 0`: `f` tuyến tính: `b >= 0` ⇒ dãy giá trị đã tăng; `b < 0` ⇒ đảo.
- `a < 0`: dùng **đối xứng dấu**: gọi chính hàm cho `(−a, −b, −c)` (có `a' > 0`), được dãy tăng của `−f`; đổi dấu và đảo.

**Giải thích từng đoạn code.**

1. `a == 0`: `vals = [b * x + c for x in nums]`; trả `vals` nếu `b >= 0`, ngược lại `vals[::-1]`.
2. `a < 0`: `res = solve(nums, -a, -b, -c)`; trả `[-v for v in reversed(res)]`.
3. `a > 0`: tính `key`, `p`, `left = [f(x) for x in nums[:p]][::-1]` (đảo thành tăng), `right = [f(x) for x in nums[p:]]`, trả `merge_two(left, right)`.

In [11]:
def sort_transformed_split(nums, a, b, c):
    if a == 0:                                          # đường thẳng
        vals = [b * x + c for x in nums]
        return vals if b >= 0 else vals[::-1]
    if a < 0:                                           # đối xứng dấu: -f có hệ số a' = -a > 0
        res = sort_transformed_split(nums, -a, -b, -c)  # dãy tăng của -f
        return [-v for v in reversed(res)]              # đổi dấu và đảo => dãy tăng của f
    # a > 0: giảm rồi tăng
    key = [2 * a * x + b for x in nums]                 # dấu của f(x+1)-f(x) ~ key (tăng theo i)
    p = bisect_left(key, 0)                             # từ p trở đi f không giảm nữa
    left = [a * x * x + b * x + c for x in nums[:p]][::-1]   # nhánh giảm -> đảo thành tăng
    right = [a * x * x + b * x + c for x in nums[p:]]        # nhánh tăng
    return merge_two(left, right)


for (nums, a, b, c), expected in examples:
    print(sort_transformed_split(nums, a, b, c), "(kỳ vọng", expected, ")")

[3, 9, 15, 33] (kỳ vọng [3, 9, 15, 33] )
[-23, -5, 1, 7] (kỳ vọng [-23, -5, 1, 7] )


In [12]:
# Code nộp LeetCode cho Cách 3
class SolutionSplit:
    def sortTransformedArray(self, nums: List[int], a: int, b: int, c: int) -> List[int]:
        def merge_two(A, B):
            out, i, j = [], 0, 0
            while i < len(A) and j < len(B):
                if A[i] <= B[j]:
                    out.append(A[i]); i += 1
                else:
                    out.append(B[j]); j += 1
            out.extend(A[i:])
            out.extend(B[j:])
            return out

        def solve(a, b, c):
            if a == 0:
                vals = [b * x + c for x in nums]
                return vals if b >= 0 else vals[::-1]
            if a < 0:
                res = solve(-a, -b, -c)
                return [-v for v in reversed(res)]
            key = [2 * a * x + b for x in nums]
            p = bisect_left(key, 0)
            left = [a * x * x + b * x + c for x in nums[:p]][::-1]
            right = [a * x * x + b * x + c for x in nums[p:]]
            return merge_two(left, right)

        return solve(a, b, c)


print(SolutionSplit().sortTransformedArray([-4, -2, 2, 4], 1, 3, 5))     # [3, 9, 15, 33]

[3, 9, 15, 33]


**Độ phức tạp Cách 3:** tạo `key` `O(n)`, `bisect_left` `O(log n)`, tính `f` `O(n)`, trộn `O(n)` ⇒ **Time `O(n)`**; **Space `O(n)`**. Cách này nặng code hơn Cách 2 nhưng cho thấy cấu trúc "hai nhánh đơn điệu" rõ ràng — hữu ích khi gặp bài tương tự mà "hai con trỏ" khó áp dụng.

### Cách 4 — Đếm (counting sort) khi miền giá trị nhỏ

**Ý tưởng.** Với ràng buộc `|x|, |a|, |b|, |c| <= 100`: `|f(x)| <= 100·100² + 100·100 + 100 = 1 010 100`. Ta dùng mảng đếm kích thước `2·1 010 100 + 1`, ghi số lần xuất hiện của mỗi giá trị, rồi duyệt mảng đếm theo thứ tự để dựng kết quả. **Không dùng quan hệ so sánh** — thuộc họ **sắp xếp không so sánh**.

**Giải thích từng đoạn.**

1. `OFFSET = 1_010_100` để chỉ số không âm: `idx = f(x) + OFFSET`.
2. `cnt[idx] += 1` cho mỗi phần tử.
3. Duyệt `idx` từ `0` đến hết; mỗi chỉ số có `cnt` lần thì thêm `cnt` lần giá trị `idx − OFFSET`.

In [13]:
OFFSET = 100 * 100 * 100 + 100 * 100 + 100              # |f(x)| lớn nhất theo ràng buộc đề


def sort_transformed_counting(nums, a, b, c):
    cnt = [0] * (2 * OFFSET + 1)
    for x in nums:
        cnt[a * x * x + b * x + c + OFFSET] += 1
    ans = []
    for idx, times in enumerate(cnt):
        if times:
            ans.extend([idx - OFFSET] * times)
    return ans


for (nums, a, b, c), expected in examples:
    print(sort_transformed_counting(nums, a, b, c), "(kỳ vọng", expected, ")")

[3, 9, 15, 33] (kỳ vọng [3, 9, 15, 33] )
[-23, -5, 1, 7] (kỳ vọng [-23, -5, 1, 7] )


**Độ phức tạp Cách 4:** `O(n + R)` với `R = 2·10^6 + 1` là kích thước miền giá trị ⇒ **Time `O(n + R)`**, **Space `O(R)`**. Với `n <= 200` thì `R` **lớn hơn nhiều** `n`, nên cách này **chậm hơn** Cách 1–3 trong bài này; nhưng ý tưởng "đếm khi miền giá trị nhỏ" là công cụ chung (`n` lớn, miền nhỏ). Cách này cũng **không** tận dụng `nums` đã sắp.

## 8. TRACE từng bước

### 8.1. Trace Cách 2 (hai con trỏ) — Ví dụ 1 (`a = 1 > 0`, điền từ cuối)

`nums = [-4,-2,2,4]`, `f(x) = x² + 3x + 5`. Mảng `ans` ban đầu `[0,0,0,0]`.

| bước | `i` | `j` | `f(nums[i])` | `f(nums[j])` | chọn | `k` | `ans` sau |
|---|---|---|---|---|---|---|---|
| 1 | 0 | 3 | `f(−4) = 9` | `f(4) = 33` | `33` (đầu phải) | 3 | `[0,0,0,33]` |
| 2 | 0 | 2 | `9` | `f(2) = 15` | `15` | 2 | `[0,0,15,33]` |
| 3 | 0 | 1 | `9` | `f(−2) = 3` | `9` (đầu trái) | 1 | `[0,9,15,33]` |
| 4 | 1 | 1 | `3` | `3` | `3` | 0 | `[3,9,15,33]` |

### 8.2. Trace Ví dụ 2 (`a = −1 < 0`, điền từ đầu)

| bước | `i` | `j` | `f(nums[i])` | `f(nums[j])` | chọn (nhỏ hơn) | `k` | `ans` sau |
|---|---|---|---|---|---|---|---|
| 1 | 0 | 3 | `−23` | `1` | `−23` | 0 | `[−23,0,0,0]` |
| 2 | 1 | 3 | `−5` | `1` | `−5` | 1 | `[−23,−5,0,0]` |
| 3 | 2 | 3 | `7` | `1` | `1` (đầu phải) | 2 | `[−23,−5,1,0]` |
| 4 | 2 | 2 | `7` | `7` | `7` | 3 | `[−23,−5,1,7]` |

In [14]:
def trace_two_pointers(nums, a, b, c):
    def f(x):
        return a * x * x + b * x + c

    n = len(nums)
    ans = [0] * n
    i, j = 0, n - 1
    mode = "lớn nhất, điền từ cuối" if a >= 0 else "nhỏ nhất, điền từ đầu"
    k = n - 1 if a >= 0 else 0
    print(f"nums = {nums}, a = {a}, b = {b}, c = {c}  -> nhặt {mode}")
    step = 0
    while i <= j:
        step += 1
        fi, fj = f(nums[i]), f(nums[j])
        if a >= 0:
            take_left = fi >= fj
        else:
            take_left = fi <= fj
        val = fi if take_left else fj
        ans[k] = val
        print(f"  bước {step}: i={i}, j={j}, f(nums[i])={fi}, f(nums[j])={fj} -> chọn {val} "
              f"({'đầu trái' if take_left else 'đầu phải'}), ans[{k}]; ans = {ans}")
        if take_left:
            i += 1
        else:
            j -= 1
        k += -1 if a >= 0 else 1
    print("kết quả:", ans)
    return ans


assert trace_two_pointers([-4, -2, 2, 4], 1, 3, 5) == [3, 9, 15, 33]
assert trace_two_pointers([-4, -2, 2, 4], -1, 3, 5) == [-23, -5, 1, 7]

nums = [-4, -2, 2, 4], a = 1, b = 3, c = 5  -> nhặt lớn nhất, điền từ cuối
  bước 1: i=0, j=3, f(nums[i])=9, f(nums[j])=33 -> chọn 33 (đầu phải), ans[3]; ans = [0, 0, 0, 33]
  bước 2: i=0, j=2, f(nums[i])=9, f(nums[j])=15 -> chọn 15 (đầu phải), ans[2]; ans = [0, 0, 15, 33]
  bước 3: i=0, j=1, f(nums[i])=9, f(nums[j])=3 -> chọn 9 (đầu trái), ans[1]; ans = [0, 9, 15, 33]
  bước 4: i=1, j=1, f(nums[i])=3, f(nums[j])=3 -> chọn 3 (đầu trái), ans[0]; ans = [3, 9, 15, 33]
kết quả: [3, 9, 15, 33]
nums = [-4, -2, 2, 4], a = -1, b = 3, c = 5  -> nhặt nhỏ nhất, điền từ đầu
  bước 1: i=0, j=3, f(nums[i])=-23, f(nums[j])=1 -> chọn -23 (đầu trái), ans[0]; ans = [-23, 0, 0, 0]
  bước 2: i=1, j=3, f(nums[i])=-5, f(nums[j])=1 -> chọn -5 (đầu trái), ans[1]; ans = [-23, -5, 0, 0]
  bước 3: i=2, j=3, f(nums[i])=7, f(nums[j])=1 -> chọn 1 (đầu phải), ans[2]; ans = [-23, -5, 1, 0]
  bước 4: i=2, j=2, f(nums[i])=7, f(nums[j])=7 -> chọn 7 (đầu trái), ans[3]; ans = [-23, -5, 1, 7]
kết quả: [-23, -5, 1, 7]


### 8.3. Trace Cách 3 (tách tại đỉnh) — Ví dụ 1

`a = 1, b = 3`: `key[i] = 2·a·nums[i] + b = 2·x + 3`: với `x = −4, −2, 2, 4` được `−5, −1, 7, 11`. `bisect_left(key, 0)` = `2` (vị trí đầu tiên có `key >= 0`). Nhánh trái `nums[:2] = [−4, −2]` ⇒ `f = [9, 3]`, đảo thành `[3, 9]`; nhánh phải `nums[2:] = [2, 4]` ⇒ `f = [15, 33]`. Trộn `[3, 9]` và `[15, 33]` ⇒ `[3, 9, 15, 33]`.

In [15]:
def trace_split(nums, a, b, c):
    assert a > 0
    key = [2 * a * x + b for x in nums]
    p = bisect_left(key, 0)
    left_raw = [a * x * x + b * x + c for x in nums[:p]]
    right = [a * x * x + b * x + c for x in nums[p:]]
    print("key        =", key)
    print("p          =", p, "(vị trí đầu tiên có key >= 0)")
    print("nhánh trái =", nums[:p], "-> f =", left_raw, "-> đảo thành tăng:", left_raw[::-1])
    print("nhánh phải =", nums[p:], "-> f =", right)
    res = merge_two(left_raw[::-1], right)
    print("trộn       =", res)
    return res


assert trace_split([-4, -2, 2, 4], 1, 3, 5) == [3, 9, 15, 33]
assert trace_split([-3, -1, 0, 5, 9], 2, -8, 1) == sorted(2 * x * x - 8 * x + 1 for x in [-3, -1, 0, 5, 9])

key        = [-5, -1, 7, 11]
p          = 2 (vị trí đầu tiên có key >= 0)
nhánh trái = [-4, -2] -> f = [9, 3] -> đảo thành tăng: [3, 9]
nhánh phải = [2, 4] -> f = [15, 33]
trộn       = [3, 9, 15, 33]
key        = [-20, -12, -8, 12, 28]
p          = 3 (vị trí đầu tiên có key >= 0)
nhánh trái = [-3, -1, 0] -> f = [43, 11, 1] -> đảo thành tăng: [1, 11, 43]
nhánh phải = [5, 9] -> f = [11, 91]
trộn       = [1, 11, 11, 43, 91]


## 9. Code nộp LeetCode

Mình nộp **Cách 2 (hai con trỏ)**: `O(n)`, ngắn, bao phủ `a > 0`, `a = 0`, `a < 0`.

In [16]:
from typing import List


class Solution:
    def sortTransformedArray(self, nums: List[int], a: int, b: int, c: int) -> List[int]:
        def f(x: int) -> int:
            return a * x * x + b * x + c

        n = len(nums)
        ans = [0] * n
        i, j = 0, n - 1
        if a >= 0:                       # chữ U hoặc đường thẳng: nhặt LỚN nhất từ hai đầu, điền từ cuối
            k = n - 1
            while i <= j:
                fi, fj = f(nums[i]), f(nums[j])
                if fi >= fj:
                    ans[k] = fi
                    i += 1
                else:
                    ans[k] = fj
                    j -= 1
                k -= 1
        else:                            # chữ ∩: nhặt NHỎ nhất từ hai đầu, điền từ đầu
            k = 0
            while i <= j:
                fi, fj = f(nums[i]), f(nums[j])
                if fi <= fj:
                    ans[k] = fi
                    i += 1
                else:
                    ans[k] = fj
                    j -= 1
                k += 1
        return ans


sol = Solution()
for (nums, a, b, c), expected in examples:
    print(sol.sortTransformedArray(nums, a, b, c), "(kỳ vọng", expected, ")")

[3, 9, 15, 33] (kỳ vọng [3, 9, 15, 33] )
[-23, -5, 1, 7] (kỳ vọng [-23, -5, 1, 7] )


## 10. Vì sao thuật toán đúng

Xét `a >= 0` (trường hợp `a < 0` đối xứng).

- **Bất biến vòng lặp:** phần `ans[k + 1 .. n − 1]` đã chứa **đúng** các giá trị lớn nhất của cả mảng, sắp tăng; đoạn `nums[i..j]` chứa các phần tử còn lại, và **mọi** giá trị `f(nums[t])` với `t` trong `[i, j]` đều **không lớn hơn** các giá trị đã điền.
- **Giá trị lớn nhất trên `[i, j]` nằm ở biên:** `f` là hàm **lồi** khi `a >= 0` (kể cả `a = 0`, tuyến tính). Với `i <= t <= j`, viết `x_t` là tổ hợp lồi của `x_i` và `x_j` (`x_t = λ·x_i + (1 − λ)·x_j`, `0 <= λ <= 1`); tính lồi cho `f(x_t) <= λ·f(x_i) + (1 − λ)·f(x_j) <= max(f(x_i), f(x_j))`. Vậy lớn nhất của đoạn là `max(f(nums[i]), f(nums[j]))`.
- **Bước chọn đúng:** ta lấy phần tử lớn hơn trong hai đầu, đặt vào `ans[k]` (vị trí cao nhất còn trống), thu đoạn ⇒ bất biến giữ nguyên.
- **Kết thúc:** `n` bước thì `ans` điền đầy; `ans` tăng dần vì mỗi lần ta điền một giá trị **không lớn hơn** giá trị điền ngay trước đó (vị trí `k` lớn hơn).
- **`a < 0`:** `f` lõm nên **nhỏ nhất** trên đoạn nằm ở biên; nhặt nhỏ nhất và điền từ đầu; chứng minh tương tự (đối xứng: `−f` lồi).
- **Số lượng phần tử trùng nhau:** `fi >= fj` hay `fi > fj` không quan trọng; khi bằng nhau lấy bên nào cũng cho cùng dãy giá trị.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Tính `f` rồi `sorted` | `O(n log n)` | `O(n)` |
| 2. Hai con trỏ (điền từ cuối/đầu) | **`O(n)`** | `O(n)` (kết quả) |
| 3. Tách tại đỉnh + trộn | `O(n)` | `O(n)` |
| 4. Đếm (counting sort) | `O(n + R)`, `R ≈ 2·10^6` | `O(R)` |

Với `n <= 200`, mọi cách đều rất nhanh; điểm của bài là **follow-up `O(n)`** (lưu ý: trong Python, `sorted` chạy bằng mã C nên trong thực tế thường nhanh hơn vòng lặp Python thuần ngay cả khi `n` lớn — bậc `O(n)` là lợi thế về lý thuyết/ngôn ngữ biên dịch) — Cách 2 và 3. Cách 4 dùng khi miền giá trị nhỏ hơn nhiều (ở đây `R` lớn nên không có lợi).

## 12. Bộ test

Oracle: `sorted(f(x) for x in nums)` (Cách 1). Hàm kiểm tra `check(nums, a, b, c)` chạy mọi cách và `Solution`, so với oracle.

In [17]:
METHODS = [sort_transformed_sorted, sort_transformed_two_pointers, sort_transformed_split,
           sort_transformed_counting,
           lambda nums, a, b, c: Solution().sortTransformedArray(nums, a, b, c),
           lambda nums, a, b, c: SolutionTwoPointers().sortTransformedArray(nums, a, b, c),
           lambda nums, a, b, c: SolutionSplit().sortTransformedArray(nums, a, b, c)]


def oracle(nums, a, b, c):
    return sorted(a * x * x + b * x + c for x in nums)


def check(nums, a, b, c, expected=None, skip_counting=False):
    want = oracle(nums, a, b, c) if expected is None else expected
    assert want == oracle(nums, a, b, c)
    for idx, f in enumerate(METHODS):
        if skip_counting and idx == 3:
            continue
        got = f(list(nums), a, b, c)
        assert got == want, (idx, nums, a, b, c, got, want)


# Ví dụ đề
check([-4, -2, 2, 4], 1, 3, 5, [3, 9, 15, 33])
check([-4, -2, 2, 4], -1, 3, 5, [-23, -5, 1, 7])

# Biên
check([0], 0, 0, 0, [0])                       # một phần tử, f = 0
check([7], 3, -2, 5)                           # một phần tử
check([-100, 100], 100, 100, 100)              # giá trị lớn nhất theo ràng buộc: 100*10^4 + ... 
check([-100, 100], -100, -100, -100)
check([-100, 0, 100], 100, 100, 100)
check([1, 2, 3, 4], 0, 0, 7, [7, 7, 7, 7])     # a = b = 0: hằng
check([1, 2, 3, 4], 0, 2, 1, [3, 5, 7, 9])     # a = 0, b > 0: giữ thứ tự
check([1, 2, 3, 4], 0, -2, 1, [-7, -5, -3, -1])   # a = 0, b < 0: đảo thứ tự
check([-3, -2, -1, 0, 1, 2, 3], 1, 0, 0, [0, 1, 1, 4, 4, 9, 9])    # bình phương (LeetCode 977)
check([-3, -2, -1, 0, 1, 2, 3], -1, 0, 0, [-9, -9, -4, -4, -1, -1, 0])
check([2, 2, 2, 2], 5, -3, 1)                  # toàn phần tử bằng nhau
check([-5, -5, 0, 0, 5, 5], 2, 0, -1)          # có trùng
check([3, 4, 5, 6], 1, -9, 0)                  # đỉnh x* = 4.5 nằm giữa (hai giá trị bằng nhau)
check([-6, -5, -4, -3], -1, -9, 0)             # đỉnh bên phải mảng, a < 0
check([1, 2, 3], 1, 0, 0, [1, 4, 9])           # toàn bộ nằm bên phải đỉnh
check([-3, -2, -1], 1, 0, 0, [1, 4, 9])        # toàn bộ nằm bên trái đỉnh (giảm)
print("test cơ bản qua")

test cơ bản qua


### 12.2. Phản ví dụ cho các cách làm sai

Ba hàm sai (mỗi hàm là một lỗi ở mục 13): `bad_only_positive_a`, `bad_abs_compare`, `bad_fill_front`, `bad_i_lt_j`.

In [18]:
def bad_only_positive_a(nums, a, b, c):
    """SAI: luôn nhặt lớn nhất, điền từ cuối, kể cả khi a < 0."""
    f = lambda x: a * x * x + b * x + c
    n = len(nums); ans = [0] * n; i, j, k = 0, n - 1, n - 1
    while i <= j:
        fi, fj = f(nums[i]), f(nums[j])
        if fi >= fj:
            ans[k] = fi; i += 1
        else:
            ans[k] = fj; j -= 1
        k -= 1
    return ans


def bad_abs_compare(nums, a, b, c):
    """SAI: so sánh |x| để chọn đầu, nghĩ 'xa gốc 0 thì f lớn hơn'."""
    f = lambda x: a * x * x + b * x + c
    n = len(nums); ans = [0] * n; i, j, k = 0, n - 1, n - 1
    while i <= j:
        if abs(nums[i]) >= abs(nums[j]):
            ans[k] = f(nums[i]); i += 1
        else:
            ans[k] = f(nums[j]); j -= 1
        k -= 1
    return ans


def bad_fill_front(nums, a, b, c):
    """SAI: a > 0 nhặt lớn nhất nhưng điền từ đầu."""
    f = lambda x: a * x * x + b * x + c
    n = len(nums); ans = [0] * n; i, j, k = 0, n - 1, 0
    while i <= j:
        fi, fj = f(nums[i]), f(nums[j])
        if fi >= fj:
            ans[k] = fi; i += 1
        else:
            ans[k] = fj; j -= 1
        k += 1
    return ans


def bad_i_lt_j(nums, a, b, c):
    """SAI: while i < j bỏ sót phần tử cuối cùng."""
    f = lambda x: a * x * x + b * x + c
    n = len(nums); ans = [0] * n; i, j, k = 0, n - 1, n - 1
    while i < j:
        fi, fj = f(nums[i]), f(nums[j])
        if fi >= fj:
            ans[k] = fi; i += 1
        else:
            ans[k] = fj; j -= 1
        k -= 1
    return ans


tests = [([-4, -2, 2, 4], -1, 3, 5), ([-4, -2, 2, 4], 1, 3, 5), ([1, 2, 3, 4, 5], 1, -6, 0)]
for bad in (bad_only_positive_a, bad_abs_compare, bad_fill_front, bad_i_lt_j):
    wrong = [t for t in tests if bad(*t) != oracle(*t)]
    print(f"{bad.__name__:22s}: sai trên {len(wrong)}/{len(tests)} test, ví dụ {wrong[0] if wrong else None} -> {bad(*wrong[0]) if wrong else None} (đúng {oracle(*wrong[0]) if wrong else None})")
    assert wrong, bad.__name__

bad_only_positive_a   : sai trên 1/3 test, ví dụ ([-4, -2, 2, 4], -1, 3, 5) -> [-23, -5, 7, 1] (đúng [-23, -5, 1, 7])
bad_abs_compare       : sai trên 3/3 test, ví dụ ([-4, -2, 2, 4], -1, 3, 5) -> [7, -5, 1, -23] (đúng [-23, -5, 1, 7])
bad_fill_front        : sai trên 3/3 test, ví dụ ([-4, -2, 2, 4], -1, 3, 5) -> [1, 7, -5, -23] (đúng [-23, -5, 1, 7])
bad_i_lt_j            : sai trên 3/3 test, ví dụ ([-4, -2, 2, 4], -1, 3, 5) -> [0, -5, 7, 1] (đúng [-23, -5, 1, 7])


### 12.3. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

Sinh `nums` đã sắp tăng (có thể trùng), `|nums[i]|, |a|, |b|, |c| <= 100`, `n` từ `1` đến `200`; cả `a < 0`, `a = 0`, `a > 0` đều xuất hiện nhiều. Cách 4 (đếm) tốn bộ nhớ nên chỉ chạy một phần các ca.

In [19]:
rng = random.Random(360)
cnt_sign = Counter()
for case in range(3000):
    n = rng.choice([1, 2, 3, 4, 5, 8, 20, 50, 200])
    span = rng.choice([3, 10, 100])
    nums = sorted(rng.randint(-span, span) for _ in range(n))
    a = rng.choice([0, rng.randint(-100, 100), rng.randint(-3, 3)])
    b = rng.choice([0, rng.randint(-100, 100), rng.randint(-3, 3)])
    c = rng.randint(-100, 100)
    cnt_sign["a>0" if a > 0 else "a<0" if a < 0 else "a=0"] += 1
    check(nums, a, b, c, skip_counting=(case % 20 != 0))
print("3000 ca ngẫu nhiên: mọi cách khớp oracle; phân bố dấu của a:", dict(cnt_sign))

3000 ca ngẫu nhiên: mọi cách khớp oracle; phân bố dấu của a: {'a=0': 1091, 'a<0': 975, 'a>0': 934}


In [20]:
# Mọi tổ hợp nhỏ (vét cạn): nums trong [-3, 3] (mọi dãy tăng có độ dài <= 4 phần tử), a, b, c trong [-3, 3]
from itertools import combinations_with_replacement
total = 0
for n in range(1, 5):
    for nums in combinations_with_replacement(range(-3, 4), n):
        for a in range(-3, 4):
            for b in range(-3, 4):
                for c in (-2, 0, 3):
                    exp = oracle(list(nums), a, b, c)
                    assert sort_transformed_two_pointers(list(nums), a, b, c) == exp
                    assert sort_transformed_split(list(nums), a, b, c) == exp
                    total += 1
print("vét cạn", total, "tổ hợp nhỏ: Cách 2 và Cách 3 đều đúng")

# Hiệu năng ở quy mô lớn hơn đề (n = 10^5) để thấy O(n) vs O(n log n)
nums = sorted(random.Random(1).randint(-10**6, 10**6) for _ in range(10**5))
for name, fn in [("sorted", sort_transformed_sorted), ("two_pointers", sort_transformed_two_pointers), ("split", sort_transformed_split)]:
    t0 = time.perf_counter()
    r = fn(nums, 3, -7, 2)
    print(f"{name:13s}: {time.perf_counter() - t0:.3f}s")
    assert r == oracle(nums, 3, -7, 2)

vét cạn 48363 tổ hợp nhỏ: Cách 2 và Cách 3 đều đúng


sorted       : 0.012s
two_pointers : 0.027s
split        : 0.016s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ nghĩ tới `a > 0`

**Code sai.** Luôn nhặt giá trị **lớn nhất** từ hai đầu và điền từ cuối, bất kể dấu của `a`.

**Vì sao sai.** Khi `a < 0` parabol mở xuống: giá trị **nhỏ nhất** ở biên, giá trị lớn nhất có thể nằm giữa. Ví dụ 2 (`a = −1`): nhặt lớn nhất từ hai đầu điền từ cuối cho `[−23, −5, 7, 1]` — không tăng dần (`7` và `1` bị đặt sai thứ tự).

**Code đúng.** Tách nhánh theo `a >= 0` / `a < 0` (nhặt lớn nhất từ cuối vs nhặt nhỏ nhất từ đầu).

In [21]:
print("sai :", bad_only_positive_a([-4, -2, 2, 4], -1, 3, 5))     # sai
print("đúng:", Solution().sortTransformedArray([-4, -2, 2, 4], -1, 3, 5))

sai : [-23, -5, 7, 1]
đúng: [-23, -5, 1, 7]


### Lỗi 2: So sánh `|x|` thay vì `f(x)`

**Code sai.** "`x` càng xa `0` thì `f` càng lớn" (đúng chỉ khi `f(x) = a·x²` với `b = 0`).

**Vì sao sai.** Đỉnh parabol là `x* = −b/(2a)`, **không** nhất thiết bằng `0`. Giá trị lớn nhất ở đầu **xa đỉnh** hơn, không phải xa `0` hơn. Ví dụ `nums = [1,2,3,4,5]`, `f(x) = x² − 6x`: đỉnh `x* = 3`, các giá trị `f` là `[−5, −8, −9, −8, −5]`. Cách sai luôn lấy đầu có `|x|` lớn hơn nên điền `−5, −8, −9, −8, −5` theo thứ tự `|x|` — giá trị nhỏ nhất `−9` (ở `x = 3`) bị đặt giữa mảng kết quả thay vì ở đầu.

**Code đúng.** Luôn so sánh **giá trị `f` đã tính** (`f(nums[i])` với `f(nums[j])`).

In [22]:
print("sai :", bad_abs_compare([1, 2, 3, 4, 5], 1, -6, 0), "| đúng:", oracle([1, 2, 3, 4, 5], 1, -6, 0))
print("sai :", bad_abs_compare([-4, -2, 2, 4], 1, 3, 5), "| đúng:", oracle([-4, -2, 2, 4], 1, 3, 5))

sai : [-5, -8, -9, -8, -5] | đúng: [-9, -8, -8, -5, -5]
sai : [15, 3, 33, 9] | đúng: [3, 9, 15, 33]


### Lỗi 3: Điền sai chiều

**Code sai.** `a > 0` nhặt lớn nhất nhưng điền `ans[0], ans[1], ...` (từ đầu).

**Vì sao sai.** Giá trị lớn nhất phải ở **cuối** mảng tăng dần. Điền từ đầu cho mảng **giảm dần**.

**Code đúng.** `a >= 0`: điền từ `k = n − 1` giảm dần; `a < 0`: nhặt nhỏ nhất, điền từ `k = 0` tăng dần. (Hoặc nhặt lớn nhất rồi `ans.reverse()` ở cuối.)

In [23]:
print("sai :", bad_fill_front([-4, -2, 2, 4], 1, 3, 5))        # [33, 15, 9, 3] giảm dần
print("đúng:", Solution().sortTransformedArray([-4, -2, 2, 4], 1, 3, 5))

sai : [33, 15, 9, 3]
đúng: [3, 9, 15, 33]


### Lỗi 4: `while i < j` bỏ sót phần tử cuối

**Code sai.** Điều kiện `i < j` thay vì `i <= j`.

**Vì sao sai.** Khi `i == j` vẫn còn **một** phần tử chưa được xử lý. Với `i < j`, vòng dừng sớm, vị trí cuối còn `0`.

**Code đúng.** `while i <= j`.

In [24]:
print("sai :", bad_i_lt_j([-4, -2, 2, 4], 1, 3, 5))             # [0, 9, 15, 33]
print("đúng:", Solution().sortTransformedArray([-4, -2, 2, 4], 1, 3, 5))

sai : [0, 9, 15, 33]
đúng: [3, 9, 15, 33]


### Lỗi 5: Dùng số thực để tìm đỉnh rồi tách mảng

**Code sai.** Tính `x_star = -b / (2 * a)`, rồi tách `nums` thành `x < x_star` và `x >= x_star`, tự xử lý `a = 0` bằng cách chia.

**Vì sao sai.** Với `a = 0`, `-b / (2 * a)` gây `ZeroDivisionError`. Với `a != 0`, so sánh số thực dễ phạm lỗi biên (đỉnh đúng bằng một phần tử), hoặc chia nguyên `-b // (2 * a)` làm tròn sai khi `a < 0`.

**Code đúng.** Tránh chia: dùng `2·a·x + b` (số nguyên) để biết dấu của `f(x + 1) − f(x)` (Cách 3), hoặc dùng hai con trỏ (Cách 2) không cần đỉnh.

In [25]:
def bad_vertex_float(nums, a, b, c):
    x_star = -b / (2 * a)                 # a = 0 -> ZeroDivisionError
    left = [x for x in nums if x < x_star]
    right = [x for x in nums if x >= x_star]
    f = lambda x: a * x * x + b * x + c
    return merge_two([f(x) for x in left][::-1], [f(x) for x in right])


try:
    bad_vertex_float([1, 2, 3], 0, 2, 1)
except ZeroDivisionError as e:
    print("sai: ZeroDivisionError:", e)
print("đúng:", Solution().sortTransformedArray([1, 2, 3], 0, 2, 1))

sai: ZeroDivisionError: division by zero
đúng: [3, 5, 7]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Áp hàm lên mảng đã sắp rồi sắp lại" và hàm là **bậc hai (lồi/lõm)** ⇒ **hai con trỏ từ hai đầu** (giống LeetCode 977 Squares of a Sorted Array: `a = 1, b = 0, c = 0`).
- Phân biệt: `a > 0` (chữ U) ⇒ lớn nhất ở **biên**; `a < 0` (chữ ∩) ⇒ nhỏ nhất ở **biên**; `a = 0` ⇒ tuyến tính (theo `b`).
- Điền kết quả theo chiều tương ứng: nhặt lớn nhất ⇒ điền **từ cuối**; nhặt nhỏ nhất ⇒ điền **từ đầu**.
- Dùng **số nguyên** để so sánh thay vì tìm đỉnh bằng số thực: `f(x + 1) − f(x) = a(2x + 1) + b`.
- Có thể đưa `a < 0` về `a > 0` bằng **đối xứng dấu**: `−f` có hệ số `(−a, −b, −c)`; kết quả của `f` = đổi dấu và đảo dãy.
- Nhớ điều kiện vòng lặp `while i <= j` (còn một phần tử vẫn phải xử lý).
- Hay bị hỏi: "làm được `O(n)` không?" → đáp: tận dụng tính **đơn điệu từng đoạn** (hai nhánh) hoặc **cực trị ở biên**.
- Khi kết quả cần đúng nhưng `n` nhỏ, `sorted(...)` luôn là câu trả lời hợp lệ và an toàn — dùng làm oracle.

## 15. Biến thể của bài

1. **Squares of a Sorted Array (LeetCode 977):** chính là bài này với `a = 1, b = 0, c = 0`.
2. **`nums` không sắp:** không còn cấu trúc ⇒ phải `sorted` (`O(n log n)`) — Cách 1.
3. **Sắp theo khoảng cách tới một điểm `k`:** `g(x) = |x − k|` là **hàm lồi** (chữ V) ⇒ cùng ý tưởng hai con trỏ, ví dụ `sort_by_distance(nums, k)`.
4. **Hàm bậc ba:** `f(x) = a·x³ + ...` có thể có tới hai điểm cực trị ⇒ tách thành **tối đa 3 đoạn đơn điệu** rồi trộn.
5. **Tìm phần tử lớn thứ `k`** sau biến đổi mà không sắp toàn bộ: dùng heap hoặc chặt nhị phân trên giá trị.
6. **Bài liên quan:** LeetCode 88 (Merge Sorted Array), 21 (Merge Two Sorted Lists), 658 (Find K Closest Elements), 167 (Two Sum II).

In [26]:
def sorted_squares(nums):
    """LeetCode 977 qua bài này: a = 1, b = 0, c = 0."""
    return Solution().sortTransformedArray(nums, 1, 0, 0)


def sort_by_distance(nums, k):
    """Sắp các giá trị |x - k| của mảng đã sắp (hàm lồi, chữ V) bằng hai con trỏ."""
    n = len(nums)
    ans = [0] * n
    i, j, pos = 0, n - 1, n - 1
    while i <= j:
        di, dj = abs(nums[i] - k), abs(nums[j] - k)
        if di >= dj:
            ans[pos] = di; i += 1
        else:
            ans[pos] = dj; j -= 1
        pos -= 1
    return ans


print(sorted_squares([-4, -1, 0, 3, 10]), sort_by_distance([-4, -1, 0, 3, 10], 2))
assert sorted_squares([-4, -1, 0, 3, 10]) == [0, 1, 9, 16, 100]
assert sorted_squares([-7, -3, 2, 3, 11]) == [4, 9, 9, 49, 121]
rng = random.Random(15)
for _ in range(300):
    nums = sorted(rng.randint(-20, 20) for _ in range(rng.randint(1, 12)))
    k = rng.randint(-25, 25)
    assert sort_by_distance(nums, k) == sorted(abs(x - k) for x in nums)

[0, 1, 9, 16, 100] [1, 2, 3, 6, 8]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tính `f` rồi `sorted` | sắp xếp thẳng | `O(n log n)` | `O(n)` | oracle; `n` nhỏ; code ngắn nhất |
| 2. Hai con trỏ | cực trị (lớn/nhỏ nhất) ở biên | `O(n)` | `O(n)` | **nên nộp**; đáp ứng follow-up |
| 3. Tách tại đỉnh + trộn | hai nhánh đơn điệu, trộn như merge | `O(n)` | `O(n)` | khi hàm có nhiều đoạn đơn điệu |
| 4. Đếm (counting sort) | đếm theo giá trị | `O(n + R)` | `O(R)` | miền giá trị nhỏ so với `n` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `f(3)` với `a = 2, b = −1, c = 4`; `sorted(x * x for x in [-3, -1, 2])`; `Solution().sortTransformedArray([0, 1, 2], 0, -1, 5)`.

**Bài 2 (tính tay).** Với `nums = [-2, 0, 1, 3]`, `a = 1, b = −2, c = 0`: tính `f` cho từng phần tử, vẽ dãy theo thứ tự `nums` (chữ U hay ∩), rồi mô phỏng hai con trỏ (bảng `i, j, f(i), f(j), chọn, ans`).

**Bài 3 (điền TODO).** Hoàn thành `sort_transformed_todo(nums, a, b, c)` theo Cách 2 (chỉ cần nhánh `a >= 0`; hàm có thể bỏ qua `a < 0`).

In [27]:
def sort_transformed_todo(nums, a, b, c):
    # TODO 1: def f(x): return a * x * x + b * x + c ; n = len(nums); ans = [0] * n; i, j, k = 0, n - 1, n - 1
    # TODO 2: while i <= j: so sánh f(nums[i]) và f(nums[j]); điền cái LỚN hơn vào ans[k]; dịch con trỏ tương ứng; k -= 1
    # TODO 3: return ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết lại **từ đầu** `sort_transformed_mine(nums, a, b, c)` cho **cả hai** dấu của `a` bằng cách: nhặt giá trị **lớn nhất** nếu `a >= 0`; nếu `a < 0` thì dùng đối xứng dấu (gọi lại với `(-a, -b, -c)` rồi đổi dấu và đảo) — không viết nhánh `k += 1`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `sort_by_distance_desc(nums, k)` trả các khoảng cách `|x − k|` sắp **giảm** dần bằng hai con trỏ. (b) Viết `kth_smallest_transformed(nums, a, b, c, k)` (k bắt đầu từ `1`) trả giá trị nhỏ thứ `k` sau biến đổi trong `O(n)` bằng cách chạy hai con trỏ chỉ `k` bước khi `a < 0` (nhặt nhỏ nhất từ đầu) hoặc `n − k + 1` bước khi `a >= 0`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `f(3) = 2·9 − 3 + 4 = 19`; `sorted([9, 1, 4]) = [1, 4, 9]`; `a = 0, b = −1`: `f = 5, 4, 3` giảm ⇒ kết quả `[3, 4, 5]`. Bài 3: ba dòng theo ba TODO.
- **Mức 2 (vừa):** Bài 2: `f(x) = x² − 2x`: `f(−2) = 8`, `f(0) = 0`, `f(1) = −1`, `f(3) = 3`: dãy `[8, 0, −1, 3]` — chữ U (đáy ở `x = 1`). Hai con trỏ nhặt lớn nhất: `(8, 3)` → `8` (trái), `(0, 3)` → `3` (phải), `(0, −1)` → `0`, `(−1, −1)` → `−1`; `ans = [−1, 0, 3, 8]`.
- **Mức 3 (sâu):** Bài 4: `if a < 0: res = mine(nums, -a, -b, -c); return [-v for v in reversed(res)]`. Bài 5(a): giống `sort_by_distance` nhưng `pos` chạy từ `0` tăng dần: nhặt khoảng cách **lớn nhất** từ hai đầu và điền từ **đầu** nên dãy giảm dần. 5(b): chạy cùng vòng lặp rồi dừng khi đã điền đủ số cần.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert 2 * 3 * 3 - 1 * 3 + 4 == 19
assert sorted(x * x for x in [-3, -1, 2]) == [1, 4, 9]
assert Solution().sortTransformedArray([0, 1, 2], 0, -1, 5) == [3, 4, 5]

# Bài 2
assert [x * x - 2 * x for x in [-2, 0, 1, 3]] == [8, 0, -1, 3]
assert Solution().sortTransformedArray([-2, 0, 1, 3], 1, -2, 0) == [-1, 0, 3, 8]


# Bài 3
def sort_transformed_todo_ans(nums, a, b, c):
    def f(x):
        return a * x * x + b * x + c

    n = len(nums)
    ans = [0] * n
    i, j, k = 0, n - 1, n - 1
    while i <= j:
        fi, fj = f(nums[i]), f(nums[j])
        if fi >= fj:
            ans[k] = fi
            i += 1
        else:
            ans[k] = fj
            j -= 1
        k -= 1
    return ans


assert sort_transformed_todo_ans([-4, -2, 2, 4], 1, 3, 5) == [3, 9, 15, 33]
assert sort_transformed_todo_ans([0, 1, 2], 0, 2, 1) == [1, 3, 5]


# Bài 4
def sort_transformed_mine(nums, a, b, c):
    if a < 0:
        res = sort_transformed_mine(nums, -a, -b, -c)
        return [-v for v in reversed(res)]
    f = lambda x: a * x * x + b * x + c
    n = len(nums)
    ans = [0] * n
    i, j, k = 0, n - 1, n - 1
    while i <= j:
        fi, fj = f(nums[i]), f(nums[j])
        if fi >= fj:
            ans[k] = fi; i += 1
        else:
            ans[k] = fj; j -= 1
        k -= 1
    return ans


rng = random.Random(19)
for _ in range(500):
    nums = sorted(rng.randint(-20, 20) for _ in range(rng.randint(1, 15)))
    a, b, c = rng.randint(-5, 5), rng.randint(-10, 10), rng.randint(-10, 10)
    assert sort_transformed_mine(nums, a, b, c) == oracle(nums, a, b, c)


# Bài 5
def sort_by_distance_desc(nums, k):
    n = len(nums)
    ans = [0] * n
    i, j, pos = 0, n - 1, 0
    while i <= j:
        di, dj = abs(nums[i] - k), abs(nums[j] - k)
        if di >= dj:
            ans[pos] = di; i += 1
        else:
            ans[pos] = dj; j -= 1
        pos += 1
    return ans


def kth_smallest_transformed(nums, a, b, c, k):
    f = lambda x: a * x * x + b * x + c
    n = len(nums)
    i, j = 0, n - 1
    if a < 0:                      # nhặt nhỏ nhất từ hai đầu: sau k lần, giá trị vừa nhặt là nhỏ thứ k
        val = None
        for _ in range(k):
            fi, fj = f(nums[i]), f(nums[j])
            if fi <= fj:
                val = fi; i += 1
            else:
                val = fj; j -= 1
        return val
    steps = n - k + 1              # a >= 0: nhặt lớn nhất; lần nhặt thứ n-k+1 chính là nhỏ thứ k
    val = None
    for _ in range(steps):
        fi, fj = f(nums[i]), f(nums[j])
        if fi >= fj:
            val = fi; i += 1
        else:
            val = fj; j -= 1
    return val


rng = random.Random(55)
for _ in range(500):
    nums = sorted(rng.randint(-15, 15) for _ in range(rng.randint(1, 12)))
    a, b, c = rng.randint(-4, 4), rng.randint(-9, 9), rng.randint(-9, 9)
    k = rng.randint(1, len(nums))
    assert kth_smallest_transformed(nums, a, b, c, k) == oracle(nums, a, b, c)[k - 1]
    kk = rng.randint(-20, 20)
    assert sort_by_distance_desc(nums, kk) == sorted((abs(x - kk) for x in nums), reverse=True)
print("Tất cả đáp án bài tập đều đúng")

Tất cả đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hai con trỏ; điền từ cuối khi `a >= 0`, từ đầu khi `a < 0`)

```python
class Solution:
    def sortTransformedArray(self, nums: List[int], a: int, b: int, c: int) -> List[int]:
        f = lambda x: a * x * x + b * x + c
        n = len(nums)
        ans = [0] * n
        i, j = 0, n - 1
        if a >= 0:
            k = n - 1
            while i <= j:
                fi, fj = f(nums[i]), f(nums[j])
                if fi >= fj:
                    ans[k] = fi; i += 1
                else:
                    ans[k] = fj; j -= 1
                k -= 1
        else:
            k = 0
            while i <= j:
                fi, fj = f(nums[i]), f(nums[j])
                if fi <= fj:
                    ans[k] = fi; i += 1
                else:
                    ans[k] = fj; j -= 1
                k += 1
        return ans
```

**Checklist**

- [ ] Phân biệt `a > 0`, `a = 0`, `a < 0` (nhánh `a >= 0` nhặt lớn nhất; `a < 0` nhặt nhỏ nhất)?
- [ ] So sánh **giá trị `f`**, không so sánh `|x|`?
- [ ] Điền đúng chiều (lớn nhất → từ cuối; nhỏ nhất → từ đầu)?
- [ ] Vòng lặp `while i <= j` (không bỏ sót phần tử cuối)?
- [ ] Trả về mảng **tăng dần** (đã thử Ví dụ 1 `[3, 9, 15, 33]` và Ví dụ 2 `[−23, −5, 1, 7]`)?
- [ ] Đã thử: `n = 1`, `a = 0` (`b > 0`, `b < 0`, `b = 0`), có phần tử trùng, đỉnh nằm giữa hai phần tử?
- [ ] Skeleton đúng `class Solution` / `def sortTransformedArray(self, nums: List[int], a: int, b: int, c: int) -> List[int]`?
- [ ] (Premium) đã đối chiếu lại đề gốc nếu có tài khoản?